In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

In [12]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"yixiaoli2000","key":"8260eb35bc4ec80df13abf4e9ad1035a"}'}

In [13]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [14]:
!kaggle competitions download -c house-prices-advanced-regression-techniques

house-prices-advanced-regression-techniques.zip: Skipping, found more recently modified local copy (use --force to force download)


In [15]:
!unzip -o house-prices-advanced-regression-techniques.zip

Archive:  house-prices-advanced-regression-techniques.zip
  inflating: data_description.txt    
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [16]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [18]:
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


1. 只做linear baseline model

In [20]:
train_df = pd.read_csv("train.csv")

num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns
num_cols = num_cols.drop("SalePrice")

X = train_df[num_cols]
y = train_df["SalePrice"]
X = X.fillna(X.mean())

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
linear_model = Ridge(alpha=1.0)
linear_model.fit(X_train, y_train)

# baselinemoxel 给xtrain预测的
y_pred_linear = linear_model.predict(X_val)

rmse_linear = np.sqrt(mean_squared_error(y_val, y_pred_linear))
print("Linear RMSE:", rmse_linear)

Linear RMSE: 36872.02907156336


In [22]:
# 第一层模型的rmse
# residual from lienar model
residuals = y_train - linear_model.predict(X_train)

In [23]:
# 有Gradient boosting
from sklearn.ensemble import GradientBoostingRegressor
nonlinear_model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05
)

nonlinear_model.fit(X_train, residuals)

GradientBoostingRegressor(learning_rate=0.05, n_estimators=200)

In [24]:
# 结合lienar和gb的组合预测
y_pred_linear = linear_model.predict(X_val)
y_pred_residual = nonlinear_model.predict(X_val) # 用gb train residual的部分
y_pred_final = y_pred_linear + y_pred_residual

rmse_final = np.sqrt(mean_squared_error(y_val, y_pred_final))

print("Linear RMSE:", rmse_linear)
print("Hybrid RMSE:", rmse_final)

Linear RMSE: 36872.02907156336
Hybrid RMSE: 35403.7259256399


结论：hybrid比linear好一点点 提成5%左右


**改进**
1. 用log y

In [25]:
y = np.log1p(train_df["SalePrice"])

2.  把catogeorial 的col也加进来


In [26]:
# 把catogeorial 的col也加进来
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.drop("SalePrice")
cat_cols = train_df.select_dtypes(include=['object']).columns

X = train_df.drop("SalePrice", axis=1)

In [28]:
# preprocessing
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")) # 空白值取mean
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # 空白值取most_frequent
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
# 再算一次linear
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

# pipeline = preprocessing了数据后的 + linear model
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_val)
rmse_linear = np.sqrt(mean_squared_error(y_val, y_pred_linear))

print("Linear RMSE:", rmse_linear)

Linear RMSE: 0.1483463436579338


In [30]:
residuals = y_train - linear_model.predict(X_train)

In [31]:
# Hybrid (ridge+gb)
from sklearn.ensemble import GradientBoostingRegressor

X_train_transformed = linear_model.named_steps["preprocessor"].transform(X_train)
X_val_transformed = linear_model.named_steps["preprocessor"].transform(X_val)

gb_model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05
)
gb_model.fit(X_train_transformed, residuals)

# combine的模型
y_pred_final_gb = y_pred_linear + gb_model.predict(X_val_transformed)
rmse_hybrid_gb = np.sqrt(mean_squared_error(y_val, y_pred_final_gb))
print("Hybrid (Ridge + GB) RMSE:", rmse_hybrid_gb)

Hybrid (Ridge + GB) RMSE: 0.14216015896590578


In [36]:
#  Hybrid(ridge+svr)
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train_transformed)
X_val_scaled = scaler.transform(X_val_transformed)

svr_model = SVR(kernel='rbf', C=50, epsilon=0.05)

svr_model.fit(X_train_scaled, residuals)

y_pred_final_svr = y_pred_linear + svr_model.predict(X_val_scaled)
rmse_hybrid_svr = np.sqrt(mean_squared_error(y_val, y_pred_final_svr))
print("Hybrid (Ridge + SVR) RMSE:", rmse_hybrid_svr)

Hybrid (Ridge + SVR) RMSE: 0.13905807454431013


接下来能做的：5-fold cross-*validation*